# Exploring ENDF/B-VIII.1

<!-- index:description -->
Explore `cross_section.xml` and HDF5 data from ENDF/B-VIII.1,
comparing direct XML/HDF5 inspection with the OpenMC Python API.
<!-- /index:description -->

<!-- generated:structure -->
```text
Exploring ENDF/B-VIII.1
├── Directory structure
├── Look inside `cross_sections.xml`
│   ├── 1. Manual parsing
│   ├── 2. Using XML parser
│   ├── 3. Using OpenMC
│   └── Comparison of the three XML interfaces
├── Look inside `HDF5` files
│   ├── Using `h5py`
│   ├── Using openmc.data.IncidentNeutron.from_hdf5()
│   └── Interpretation of MT numbers
└── Materials × MT availability
```
<!-- /generated:structure -->

The data location is configured in [openmc_workbench_config.py](../openmc_workbench_config.py). Before running this notebook, install the shared module into your Jupyter kernel’s Python environment: run `%pip install -e /absolute/path/to/openmc-workbench` in a notebook cell, replacing the path with your repository checkout. The default installation is `~/data`; edit `DATA_ROOT` in the shared module to use another location. Restart the kernel after changing the configuration.

**Scope of this notebook**

This notebook explores how OpenMC nuclear-data libraries are organized and accessed, both directly through their underlying XML/HDF5 files and through the OpenMC Python API.

Imports are introduced at the point of use to highlight the dependencies specific to each plotting approach.

The manual methods are included for **exploratory and educational purposes**, to expose the underlying file structures and illustrate how the higher-level interfaces work. They are not intended as production implementations.

For production applications, prefer established parsers and the OpenMC API, which provide better abstraction, validation, and compatibility with supported data formats. Direct access to XML/HDF5 remains appropriate when the task specifically requires inspecting or manipulating the underlying data structures.

## Directory structure

In [1]:
from pathlib import Path

from openmc_workbench_config import ENDF_B81

nuc_dat_path = ENDF_B81
assert nuc_dat_path.is_dir()
for tis in Path(nuc_dat_path).iterdir():
    if tis.is_file():
        print(tis)
# Top-level directory contains just a xml file

../../data/endf-b8.1/cross_sections.xml


This will be the argument for [openmc.data.DataLibrary.from_xml()](https://docs.openmc.org/en/stable/pythonapi/generated/openmc.data.DataLibrary.html#openmc.data.DataLibrary).

Assign it to `xs_xml`.

In [2]:
xs_xml = nuc_dat_path / 'cross_sections.xml'
assert xs_xml.is_file()

`cross_sections.xml` reminds me of MCNP's `xsdir`: they provide index lookup.

|                      | MCNP's xsdir              | OpenMC's cross_sections.xml |
| --- | --- | --- |
| Index file             | `xsdir`                   | `cross_sections.xml` |
| Index format           | Plain text                | XML |
| Nuclear-data format    | ACE                       | HDF5 |
| Nuclide identification | ZAID                      | `materials` attribute | 
| Data type              | ZAID suffix e.g. `.80c` | `type` attribute |

In [3]:
for tis in Path(nuc_dat_path).iterdir():
    if tis.is_dir():
        print(tis)
# Top-level directory contains these subdirectories.

../../data/endf-b8.1/thermal
../../data/endf-b8.1/neutron
../../data/endf-b8.1/photon


In [4]:
max_file_depth = 0
for path in nuc_dat_path.rglob('*'):
    if path.is_file():
        depth = len(path.relative_to(nuc_dat_path).parts)
        max_file_depth = max(max_file_depth, depth)
max_file_depth

2

## Look inside `cross_sections.xml`

Ways of looking inside:
1. manual parsing line by line;
2. using XML parser `xml.etree.ElementTree`;
3. using OpenMC.

### 1. Manual parsing
The regular expression relies on XML attribute order and whitespace; this is intentional manual parsing.

In [5]:
import re
import pandas as pd          # Hard to live without pandas as an ML person.

xs_manual = []               # This will become a dataframe when loop exits, just borrowing a list's efficiency.
with open(xs_xml) as fp:
    lines = fp.read().splitlines()
    for nline, line in enumerate(lines):
        match = re.search(
            r'  <library materials="(.*)" path="(.*)" type="(.*)" />',
            line
        )
        
        if match:
            xs_manual.append(match.groups())
        else:
            print(f'{nline}/{len(lines)}', line)

xs_manual = pd.DataFrame(xs_manual, columns=['materials', 'path', 'type'])
assert (                                        # `type` consistently agrees with the subdirectory name: no surprises.
    xs_manual['path'].map(lambda x: Path(x).parts[0])
    == xs_manual['type']
).all() 
xs_manual
# Apart from the header row, and the start and end `cross_sections` tags,
# all remaining rows have the same pattern: <library materials=... path=... type=.../>.

0/771 <?xml version='1.0' encoding='utf-8'?>
1/771 <cross_sections>
770/771 </cross_sections>


,materials,path,type
0,H1,neutron/H1.h5,neutron
1,H2,neutron/H2.h5,neutron
2,H3,neutron/H3.h5,neutron
3,He3,neutron/He3.h5,neutron
4,He4,neutron/He4.h5,neutron
...,...,...,...
763,Cm,photon/Cm.h5,photon
764,Bk,photon/Bk.h5,photon
765,Cf,photon/Cf.h5,photon
766,Es,photon/Es.h5,photon


In [6]:
assert xs_manual['path'].map(lambda x: (nuc_dat_path / x).is_file()).all()
# Every single entry corresponds to an HDF5 file (extension `.h5`) in the respective subdirectory.

### 2. Using XML parser

In [7]:
import xml.etree.ElementTree as ET

root = ET.parse(xs_xml).getroot()

xs_etree = pd.DataFrame([
    element.attrib
    for element in root.findall('library')
])


### 3. Using OpenMC

In [8]:
import openmc

library = openmc.data.DataLibrary.from_xml(xs_xml)
for tis in library.libraries:
    assert isinstance(tis, dict)
    assert {'materials', 'path', 'type'} <= tis.keys()
# Consistent with what we found when looking inside the xml: 
# it contains simple length-of-three dictionaries. No nesting.
library.libraries

[{'path': '../../data/endf-b8.1/neutron/H1.h5',
  'type': 'neutron',
  'materials': ['H1']},
 {'path': '../../data/endf-b8.1/neutron/H2.h5',
  'type': 'neutron',
  'materials': ['H2']},
 {'path': '../../data/endf-b8.1/neutron/H3.h5',
  'type': 'neutron',
  'materials': ['H3']},
 {'path': '../../data/endf-b8.1/neutron/He3.h5',
  'type': 'neutron',
  'materials': ['He3']},
 {'path': '../../data/endf-b8.1/neutron/He4.h5',
  'type': 'neutron',
  'materials': ['He4']},
 {'path': '../../data/endf-b8.1/neutron/Li6.h5',
  'type': 'neutron',
  'materials': ['Li6']},
 {'path': '../../data/endf-b8.1/neutron/Li7.h5',
  'type': 'neutron',
  'materials': ['Li7']},
 {'path': '../../data/endf-b8.1/neutron/Be7.h5',
  'type': 'neutron',
  'materials': ['Be7']},
 {'path': '../../data/endf-b8.1/neutron/Be9.h5',
  'type': 'neutron',
  'materials': ['Be9']},
 {'path': '../../data/endf-b8.1/neutron/B10.h5',
  'type': 'neutron',
  'materials': ['B10']},
 {'path': '../../data/endf-b8.1/neutron/B11.h5',
  'type

In [9]:
xs_openmc = pd.DataFrame(library.libraries)
xs_openmc['materials'] = xs_openmc['materials'].map(lambda x: x[0])
xs_openmc['path'] = xs_openmc['path'].map(lambda x: str(Path(x).relative_to(nuc_dat_path)))
xs_openmc = xs_openmc[xs_etree.columns]
xs_openmc

,materials,path,type
0,H1,neutron/H1.h5,neutron
1,H2,neutron/H2.h5,neutron
2,H3,neutron/H3.h5,neutron
3,He3,neutron/He3.h5,neutron
4,He4,neutron/He4.h5,neutron
...,...,...,...
763,Cm,photon/Cm.h5,photon
764,Bk,photon/Bk.h5,photon
765,Cf,photon/Cf.h5,photon
766,Es,photon/Es.h5,photon


### Comparison of the three XML interfaces

The three approaches—manual text parsing, Python's `xml.etree.ElementTree`, and OpenMC's `DataLibrary`—produce equivalent tables for the particular `cross_sections.xml` file examined here, after normalizing their representations.

This equivalence is not guaranteed for arbitrary XML files. The manual parser relies on the file's formatting and attribute order. `ElementTree` interprets XML according to its structure, while OpenMC's `DataLibrary` converts certain attributes into Python objects and resolves library paths.

For example, OpenMC represents `materials` as a list and library paths may be resolved relative to the XML file's location. The comparison therefore requires converting these values back into the representation used in the original XML file.

The equality assertions validate the results for this dataset; they do not establish that the three methods are universally interchangeable.

In [10]:
# Confirm `xs_manual`==`xs_etree`==`xs_openmc`:
pd.testing.assert_frame_equal(xs_manual, xs_etree) 
pd.testing.assert_frame_equal(xs_manual, xs_openmc) 
# Rename as `df`, remove dataframes we no longer need.
df = xs_etree
del xs_manual, xs_etree, xs_openmc

## Look inside `HDF5` files
HDF5 files are binary files. HDF is an acronym for *Hierarchical Data Format*.
The [format](https://www.hdfgroup.org/solutions/hdf5/) is cross platform, with no vendor lock-in.
Ways of looking inside:
1. using `h5py`;
2. using [openmc.data.IncidentNeutron.from_hdf5()](https://docs.openmc.org/en/stable/pythonapi/generated/openmc.data.IncidentNeutron.html).

### Using `h5py`

In [11]:
import h5py

# First, try with a single material.
pic = (df['materials']=='Li6') & (df['type']=='neutron')
filename = df.loc[pic, 'path'].iloc[0]

with h5py.File(nuc_dat_path / filename) as ff:
    print(list(ff.keys()))
    ff.visit(print)
# A single nuclide's HDF5 file contains neutron cross sections at multiple temperatures.

['Li6']
Li6
Li6/energy
Li6/energy/0K
Li6/energy/1200K
Li6/energy/2500K
Li6/energy/250K
Li6/energy/294K
Li6/energy/600K
Li6/energy/900K
Li6/kTs
Li6/kTs/1200K
Li6/kTs/2500K
Li6/kTs/250K
Li6/kTs/294K
Li6/kTs/600K
Li6/kTs/900K
Li6/reactions
Li6/reactions/reaction_002
Li6/reactions/reaction_002/0K
Li6/reactions/reaction_002/0K/xs
Li6/reactions/reaction_002/1200K
Li6/reactions/reaction_002/1200K/xs
Li6/reactions/reaction_002/2500K
Li6/reactions/reaction_002/2500K/xs
Li6/reactions/reaction_002/250K
Li6/reactions/reaction_002/250K/xs
Li6/reactions/reaction_002/294K
Li6/reactions/reaction_002/294K/xs
Li6/reactions/reaction_002/600K
Li6/reactions/reaction_002/600K/xs
Li6/reactions/reaction_002/900K
Li6/reactions/reaction_002/900K/xs
Li6/reactions/reaction_002/product_0
Li6/reactions/reaction_002/product_0/distribution_0
Li6/reactions/reaction_002/product_0/distribution_0/angle
Li6/reactions/reaction_002/product_0/distribution_0/angle/energy
Li6/reactions/reaction_002/product_0/distribution_0/ang

In [12]:
# Next, loop over all materials
def get_reactions(filename):
    with h5py.File(nuc_dat_path / filename) as ff:
        nuclide_name = list(ff.keys())[0]
        reactions = ff[nuclide_name]['reactions']

        mts = []
        for name in reactions:
            if name.startswith('reaction_'):
                mt = int(name.removeprefix('reaction_'))
                mts.append(mt)

    return sorted(mts)
df['MTs'] = None
pic = df['type']=='neutron'
df.loc[pic, 'MTs'] = df.loc[pic, 'path'].map(lambda x: get_reactions(x))
df

,materials,path,type,MTs
0,H1,neutron/H1.h5,neutron,"[2, 102, 204, 301, 444, 901]"
1,H2,neutron/H2.h5,neutron,"[2, 16, 102, 203, 205, 301, 444, 901]"
2,H3,neutron/H3.h5,neutron,"[2, 16, 204, 301, 444, 901]"
3,He3,neutron/He3.h5,neutron,"[2, 102, 103, 104, 203, 204, 205, 207, 301, 44..."
4,He4,neutron/He4.h5,neutron,"[2, 301, 444, 901]"
...,...,...,...,...
763,Cm,photon/Cm.h5,photon,None
764,Bk,photon/Bk.h5,photon,None
765,Cf,photon/Cf.h5,photon,None
766,Es,photon/Es.h5,photon,None


### Using openmc.data.IncidentNeutron.from_hdf5()

In [13]:
# Demonstrating with just one material.
# Looping through every material would be too CPU-expensive.
library.get_by_material("Li6")

{'path': '../../data/endf-b8.1/neutron/Li6.h5',
 'type': 'neutron',
 'materials': ['Li6']}

In [14]:
pic = (df['materials']=='Li6') & (df['type']=='neutron')
filename = f"{nuc_dat_path}/{df.loc[pic, 'path'].values[0]}"
data = openmc.data.IncidentNeutron.from_hdf5(filename)
mts_openmc = set(data.reactions)
mts_openmc

{np.int64(2),
 np.int64(32),
 np.int64(41),
 np.int64(52),
 np.int64(102),
 np.int64(103),
 np.int64(105),
 np.int64(203),
 np.int64(204),
 np.int64(205),
 np.int64(207),
 np.int64(301),
 np.int64(444),
 np.int64(901)}

In [15]:
# That matches what we compiled (manually) from `HDF5`, in `df`.
mts_hdf5 = set(df.loc[pic, 'MTs'].values[0])
assert mts_openmc == mts_hdf5

### Interpretation of MT numbers

ENDF MT numbers identify different types of nuclear-reaction information. Many correspond to physical reaction channels, such as elastic scattering (MT 2), radiative capture (MT 102), and neutron-induced triton production, (n,t) (MT 105).

However, not every MT identifier represents a distinct physical reaction channel. Some identify derived or processed quantities, including heating or energy-deposition information (MT 301) and damage-energy production (MT 444).

Consequently, the list of MT identifiers stored in an OpenMC HDF5 file should not be interpreted simply as a list of independent physical reactions. It may also contain quantities calculated from or associated with nuclear reactions.

In [16]:
def get_reaction_names(mts):
    if mts is None:
        return None
    return [
        openmc.data.REACTION_NAME.get(mt, 'Unknown')
        for mt in mts
    ]

df['reaction_names'] = df['MTs'].map(get_reaction_names)
df

,materials,path,type,MTs,reaction_names
0,H1,neutron/H1.h5,neutron,"[2, 102, 204, 301, 444, 901]","[(n,elastic), (n,gamma), (n,Xd), heating, dama..."
1,H2,neutron/H2.h5,neutron,"[2, 16, 102, 203, 205, 301, 444, 901]","[(n,elastic), (n,2n), (n,gamma), (n,Xp), (n,Xt..."
2,H3,neutron/H3.h5,neutron,"[2, 16, 204, 301, 444, 901]","[(n,elastic), (n,2n), (n,Xd), heating, damage-..."
3,He3,neutron/He3.h5,neutron,"[2, 102, 103, 104, 203, 204, 205, 207, 301, 44...","[(n,elastic), (n,gamma), (n,p), (n,d), (n,Xp),..."
4,He4,neutron/He4.h5,neutron,"[2, 301, 444, 901]","[(n,elastic), heating, damage-energy, heating-..."
...,...,...,...,...,...
763,Cm,photon/Cm.h5,photon,None,None
764,Bk,photon/Bk.h5,photon,None,None
765,Cf,photon/Cf.h5,photon,None,None
766,Es,photon/Es.h5,photon,None,None


## Materials × MT availability

Rows are neutron-data materials and columns are MT numbers.
`True` indicates that the material's HDF5 file contains that MT; `False` indicates it does not.

In [17]:
# Expand the MT lists, then pivot into a material-by-MT presence table.
pic = df['type']=='neutron'
neutron_mts = df.loc[pic, ['materials', 'MTs']]
neutron_mts

,materials,MTs
0,H1,"[2, 102, 204, 301, 444, 901]"
1,H2,"[2, 16, 102, 203, 205, 301, 444, 901]"
2,H3,"[2, 16, 204, 301, 444, 901]"
3,He3,"[2, 102, 103, 104, 203, 204, 205, 207, 301, 44..."
4,He4,"[2, 301, 444, 901]"
...,...,...
552,Es253,"[2, 16, 17, 18, 37, 51, 52, 53, 54, 55, 91, 10..."
553,Es254,"[2, 16, 17, 18, 37, 51, 52, 53, 54, 55, 56, 57..."
554,Es254_m1,"[2, 16, 17, 18, 37, 51, 52, 53, 54, 55, 56, 57..."
555,Es255,"[2, 16, 17, 18, 37, 51, 52, 91, 102, 301, 444,..."


In [18]:
material_mt_pairs = neutron_mts.explode('MTs')
material_mt_pairs.rename(columns={'MTs': 'MT'}, inplace=True)
material_mt_pairs['MT'] = material_mt_pairs['MT'].astype(int)
material_mt_pairs['available'] = True
material_mt_pairs

,materials,MT,available
0,H1,2,True
0,H1,102,True
0,H1,204,True
0,H1,301,True
0,H1,444,True
...,...,...,...
556,Fm255,91,True
556,Fm255,102,True
556,Fm255,301,True
556,Fm255,444,True


In [19]:
material_mt_table = material_mt_pairs.pivot_table(
        index='materials', columns='MT', values='available', 
        aggfunc='any', fill_value=False)
material_mt_table

MT,2,3,4,5,11,16,17,18,19,20,...,832,833,834,835,836,837,838,839,849,901
materials,,,,,,,,,,,,,,,,,,,,,
Ac225,True,False,False,False,False,True,True,True,False,False,...,False,False,False,False,False,False,False,False,False,True
Ac226,True,False,False,False,False,True,True,True,False,False,...,False,False,False,False,False,False,False,False,False,True
Ac227,True,False,False,False,False,True,True,True,False,False,...,False,False,False,False,False,False,False,False,False,True
Ag107,True,False,True,False,False,True,True,False,False,False,...,False,False,False,False,False,False,False,False,True,True
Ag108,True,False,False,True,False,True,False,False,False,False,...,True,True,True,True,True,True,True,True,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Zr92,True,False,False,True,False,True,True,False,False,False,...,True,True,True,True,True,True,True,True,True,True
Zr93,True,False,False,True,False,True,True,False,False,False,...,False,False,False,False,False,False,False,False,True,True
Zr94,True,False,False,True,False,True,True,False,False,False,...,True,True,True,True,True,True,True,True,True,True
